# Secure AI Agent Access with Amazon Bedrock AgentCore Identity

In this notebook, you secure an IT help desk agent using Amazon Bedrock AgentCore Identity's **inbound auth** capability. You deploy the agent to AgentCore Runtime so that only callers presenting a valid Amazon Cognito JWT can invoke it — everyone else is rejected before the request reaches your code.

# Task 2: Build and Test the Agent Locally

In this task, you build and run the IT help desk agent locally to confirm it works. The IT help desk team at AnyCompany built this agent to answer common employee IT questions.

### Task 2.1: Import Libraries

In this task, you import the libraries needed to build and run the agent locally. 

To run the cell below, select within the cell and press **Shift + Enter** or, at the top of the page, choose the **Run** button.

In [ ]:
from boto3.session import Session

from strands import Agent, tool
from strands.models import BedrockModel

boto_session = Session()
REGION = boto_session.region_name

MODEL_ID = "us.amazon.nova-pro-v1:0"

print(f"✅ Region: {REGION}\n")
print(f"Your agent will use the following model for reasoning:")
print(f"✅ Model: {MODEL_ID}\n")

### Task 2.2: Build the Agent Locally

In this task, you build the same agent that runs inside the container. It's a Strands agent with two tools: `lookup_internal_kb` (searches a local help desk knowledge base) and `create_support_ticket` (opens a simulated ticket when the KB can't resolve the issue).

There is **no authentication logic** anywhere in this code. The agent just processes prompts. You add security at the Runtime layer in Task 3.

In [ ]:
INTERNAL_KB = {
    "vpn":     "VPN troubleshooting: 1) Restart the GlobalProtect client. 2) Verify network connectivity. 3) Check for duplicate sessions. 4) Contact IT if the issue persists.",
    "email":   "Email access: Use Outlook Web for browser-based access. For mobile, install the Outlook mobile app and sign in with your corporate credentials.",
    "wifi":    "Corporate Wi-Fi: Connect to 'AnyCompany-Corp'. Sign in with your corporate username and password. Off-site, use the VPN.",
    "sso":     "Single sign-on: SSO is provided through the company identity provider. If you receive a 'session expired' error, sign out of all corporate apps and sign in again.",
    "printer": "Printer setup: Add the printer queue named 'corp-print-01.anycompany.local'. Use 'Print to PDF' if a physical printer is unavailable.",
}


@tool
def lookup_internal_kb(topic: str) -> str:
    """Look up a help desk knowledge base entry. Topics: vpn, email, wifi, sso, printer."""
    key = topic.strip().lower()
    if key in INTERNAL_KB:
        return INTERNAL_KB[key]
    return f"No KB entry found for '{topic}'. Available topics: {', '.join(INTERNAL_KB.keys())}."


_TICKET_COUNTER = 1000


@tool
def create_support_ticket(summary: str, priority: str = "medium") -> str:
    """Open an IT help desk support ticket when an issue can't be resolved from the KB.

    Args:
        summary: A short description of the employee's issue.
        priority: Ticket priority — one of: low, medium, high. Defaults to medium.
    """
    global _TICKET_COUNTER
    valid_priorities = {"low", "medium", "high"}
    level = priority.strip().lower()
    if level not in valid_priorities:
        level = "medium"

    _TICKET_COUNTER += 1
    ticket_id = f"INC-{_TICKET_COUNTER}"
    return (
        f"Support ticket created. Ticket ID: {ticket_id} | "
        f"Priority: {level} | Summary: {summary.strip()}. "
        "An IT support agent will follow up shortly."
    )


model = BedrockModel(model_id=MODEL_ID, temperature=0.3, region_name=REGION)

agent = Agent(
    model=model,
    tools=[lookup_internal_kb, create_support_ticket],
    system_prompt=(
        "You are an internal IT help desk assistant for AnyCompany. "
        "Use the lookup_internal_kb tool to answer common employee IT questions. "
        "If the knowledge base doesn't resolve the issue, use the create_support_ticket "
        "tool to open a ticket for follow-up. Be concise and helpful."
    ),
)

print("✅ Agent built successfully")

### Task 2.3: Test the Agent

In this task, you invoke the agent to confirm it answers correctly.

In [ ]:
print("💬 Testing the agent locally...\n")
print("-" * 60)
print("🤖 Agent response:")
response = agent("How do I get on the corporate Wi-Fi?")
print("\n" + "=" * 60)

**Task complete.** You built and tested the IT help desk agent locally.

# Task 3: Deploy the Agent to AgentCore Runtime with AgentCore Identity Inbound Auth

In this task, you deploy the agent to AgentCore Runtime and turn on inbound auth. Inbound auth is the AgentCore Identity capability that checks a caller's identity before a request reaches your agent. 

### Task 3.1: Read the Lab Configuration

The lab has already set up the supporting pieces for you — a Cognito user pool with an app client and a confirmed test user, the pre-built agent container image, and the Runtime's execution role. In this task, you load their IDs from AWS Systems Manager (SSM) Parameter Store and use them to deploy and secure the agent in the tasks ahead.

In [ ]:
import json
import time
import boto3

from lab_helpers.utils import (
    get_ssm_parameter,
    cognito_authenticate,
    decode_jwt_unsafe,
)

# Pre-provisioned configuration loaded from SSM
COGNITO_USER_POOL_ID = get_ssm_parameter("/app/agentcore-identity-lab/cognito_user_pool_id")
COGNITO_CLIENT_ID    = get_ssm_parameter("/app/agentcore-identity-lab/cognito_client_id")
COGNITO_USERNAME     = get_ssm_parameter("/app/agentcore-identity-lab/cognito_test_username")
COGNITO_PASSWORD     = get_ssm_parameter("/app/agentcore-identity-lab/cognito_test_user_password")
COGNITO_DISCOVERY_URL = get_ssm_parameter("/app/agentcore-identity-lab/cognito_discovery_url")

# Pre-built agent container and its execution role
ECR_IMAGE_URI       = get_ssm_parameter("/app/agentcore-identity-lab/ecr_image_uri")
EXECUTION_ROLE_ARN  = get_ssm_parameter("/app/agentcore-identity-lab/execution_role_arn")

print(f"✅ Cognito user pool: {COGNITO_USER_POOL_ID}")
print(f"✅ Cognito discovery URL: {COGNITO_DISCOVERY_URL}")
print(f"✅ Cognito test user: {COGNITO_USERNAME}")
print(f"✅ ECR image: {ECR_IMAGE_URI}")

### Task 3.2: Create the AgentCore Runtime with Inbound Auth

In this task, you create the AgentCore Runtime and turn on inbound auth using `CreateAgentRuntime`. Run the cell below to deploy the agent.

In [ ]:
agentcore_client = boto3.client("bedrock-agentcore-control", region_name=REGION)

print("🚀 Creating AgentCore Runtime with inbound auth...\n")

create_response = agentcore_client.create_agent_runtime(
    agentRuntimeName="helpdesk_agent_inbound_auth",
    agentRuntimeArtifact={
        # Point the Runtime at the pre-built agent image in ECR. The Runtime
        # pulls this image and runs it; the URI was loaded from SSM in Task 3.1.
        "containerConfiguration": {
            "containerUri": ECR_IMAGE_URI,
        }
    },
    roleArn=EXECUTION_ROLE_ARN,
    networkConfiguration={
        "networkMode": "PUBLIC",
    },
    # Inbound auth — the part that secures the agent. Only callers presenting a
    # valid Cognito JWT are allowed through; everyone else is rejected.
    authorizerConfiguration={
        "customJWTAuthorizer": {
            # Cognito's OpenID Connect endpoint, where AgentCore Identity fetches the signing keys.
            "discoveryUrl": COGNITO_DISCOVERY_URL,
            # Only JWTs issued to these app client IDs are accepted.
            "allowedClients": [COGNITO_CLIENT_ID],
        }
    },
)

AGENT_RUNTIME_ID = create_response["agentRuntimeId"]
AGENT_RUNTIME_ARN = create_response["agentRuntimeArn"]

print(f"✅ Runtime creation initiated")
print(f"   Agent Runtime ID: {AGENT_RUNTIME_ID}")
print(f"   Agent Runtime ARN: {AGENT_RUNTIME_ARN}")
print(f"   Authorizer: customJWTAuthorizer")
print(f"   Discovery URL: {COGNITO_DISCOVERY_URL}")
print(f"   Allowed clients: [{COGNITO_CLIENT_ID}]")
print(f"\n⏳ Deployment is provisioning in the background. Proceed to Task 4 to invoke it — the first step waits for the deployment to finish.")

**How the code works.** The parameter that secures the agent is **`authorizerConfiguration`** — it turns on inbound auth by adding a `customJWTAuthorizer` to the Runtime, which tells AgentCore Identity to validate each request's JWT (JSON Web Token) against Cognito before allowing it through. Two fields define that check: `discoveryUrl` is a Cognito endpoint where AgentCore Identity fetches the keys to verify a token's signature, and `allowedClients` is the list of Cognito app client IDs whose tokens are accepted — anything else is rejected.

`containerUri` points to your agent's image in Amazon ECR — the same Task 2 agent, containerized so the Runtime can run it. The ECR image was pre-built by the lab so you can stay focused on inbound auth.



The following image shows what happens during a deployment on AgentCore Runtime:

![The agent goes from local code to a container image in Amazon ECR, then to a deployment on AgentCore Runtime.](images/invoke.png)
*Image description: The agent goes from local code to a container image in Amazon ECR, then to a deployment on AgentCore Runtime. Once deployed to AgentCore Runtime, the agent is ready to be invoked by an end user — which you'll do in the next task.*


**Task complete.** You created the AgentCore Runtime with inbound auth enabled. It finishes provisioning in the background while you move on to Task 4.

# Task 4: Invoke the Deployed Agent

In this task, you confirm the agent is ready, then test inbound auth from both sides: a request without a valid JWT is rejected, and a request with one reaches the agent.

A *JWT* (JSON Web Token) is the signed credential the caller attaches to each request as proof of identity. The caller sends it in the HTTP `Authorization` header, and AgentCore Identity validates it before the request reaches your agent.

### Task 4.1: Check Deployment Status

In this task, you wait for the Runtime to reach READY status.

In [ ]:
print("⏳ Checking deployment status...")

# Terminal statuses for a CreateAgentRuntime deployment. Valid AgentCore Runtime statuses are:
# CREATING | CREATE_FAILED | UPDATING | UPDATE_FAILED | READY | DELETING
end_statuses = ["READY", "CREATE_FAILED", "UPDATE_FAILED"]
max_wait_seconds = 600
waited = 0

status_response = agentcore_client.get_agent_runtime(agentRuntimeId=AGENT_RUNTIME_ID)
status = status_response["status"]

while status not in end_statuses and waited < max_wait_seconds:
    print(f"   Status: {status} — waiting...")
    time.sleep(15)
    waited += 15
    status_response = agentcore_client.get_agent_runtime(agentRuntimeId=AGENT_RUNTIME_ID)
    status = status_response["status"]

if status == "READY":
    print(f"\n✅ Agent Runtime is READY")
    print(f"   Runtime ID: {AGENT_RUNTIME_ID}")
elif status in end_statuses:
    print(f"\n❌ Deployment ended with status: {status}")
    print("   Check the AgentCore console for details.")
else:
    print(f"\n⏱️  Still '{status}' after {max_wait_seconds}s. Re-run this cell to keep waiting, or check the AgentCore console.")

### Task 4.2: Invoke Without a JWT

In this task, you invoke the agent without a JWT. AgentCore Identity rejects the request with an access-denied error (an `AccessDeniedException`) — your agent code never even runs.

In [ ]:
agentcore_data_client = boto3.client("bedrock-agentcore", region_name=REGION)


def read_agent_response(response):
    """Read an InvokeAgentRuntime response, handling streaming and non-streaming."""
    content_type = response.get("contentType", "")
    body = response["response"]
    if "text/event-stream" in content_type:
        parts = []
        for event in body:
            raw = event if isinstance(event, bytes) else event.get("chunk", {}).get("bytes", b"")
            if raw:
                parts.append(raw.decode("utf-8"))
        return "".join(parts)
    return json.loads(body.read())


print("💬 Invoking the agent WITHOUT a JWT...\n")
try:
    # A JWT would go in the HTTP Authorization header, but neither boto3 nor the code below adds one — so the request arrives with no credential.
    response = agentcore_data_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        payload=json.dumps({"prompt": "How do I get on the corporate Wi-Fi?"}),
    )
    print(f"⚠️  Unexpected success: {response}")
except agentcore_data_client.exceptions.AccessDeniedException as e:
    # Expected result: the JWT authorizer rejected the unauthenticated call.
    print(f"✅ Rejected by AgentCore Identity with the following error message:")
    print(f"\n{e}")
    print("\nNo valid JWT was provided, so the request never reached your agent.")
except Exception as e:
    # Any other error is NOT the auth check — don't mistake it for a successful rejection.
    print(f"⚠️  Unexpected error (not the auth rejection we expected): {type(e).__name__}: {e}")
    print("\n   Confirm the Runtime reached READY in Task 4.1, then re-run this cell.")

### Task 4.3: Get a JWT from Cognito

In this task, you authenticate to the Cognito user pool to get a fresh JWT. You inspect it in the next task, then use it to invoke the agent.

In [ ]:
# Authenticate to Cognito to get a JWT (returned in the access_token field)
tokens = cognito_authenticate(
    client_id=COGNITO_CLIENT_ID,
    username=COGNITO_USERNAME,
    password=COGNITO_PASSWORD,
    region=REGION,
)
jwt = tokens["access_token"]

print("✅ Retrieved a JWT from Cognito:\n")
print(jwt)

### Task 4.4: Look Inside the JWT

Before you invoke the agent, decode the JWT you just retrieved to see what it actually contains. This decodes the token locally for inspection — it does not verify it (that's AgentCore Identity's job when you invoke next).

In [ ]:
decoded = decode_jwt_unsafe(jwt)

print("📋 JWT header:")
print(json.dumps(decoded["header"], indent=2))
print()
print("📋 JWT payload (claims):")
for k in ("iss", "client_id", "sub", "exp", "iat", "token_use", "scope", "username"):
    if k in decoded["payload"]:
        print(f"  {k}: {decoded['payload'][k]}")
print()
print("📋 Signature (first 20 chars):")
print(f"  {decoded['signature_preview']}")
print()
print("=" * 60)
print()
print("🔑 How these claims map to your Runtime configuration:")
print(f"  iss (issuer)  → Must match the issuer published at your discovery URL")
print(f"  client_id     → Must be in allowedClients: [{COGNITO_CLIENT_ID}]")
print(f"  exp (expiry)  → AgentCore Identity rejects expired JWTs automatically")
print(f"  signature     → Verified against keys fetched from: {COGNITO_DISCOVERY_URL}")

For this lab's authorizer configuration, these are the claims AgentCore Identity checks when you invoke the agent — issuer, client_id, signature, and expiration. If any one fails, it rejects the call, just like the request with no JWT. Next, you invoke the agent with this JWT and watch these checks pass.

### Task 4.5: Invoke With the Valid JWT

In this task, you invoke the agent with the JWT. AgentCore Identity validates it — checks signature, issuer, client_id, expiration — and forwards the request to your agent.

In [ ]:
print("💬 Invoking the agent WITH a valid JWT...\n")
print("-" * 60)

# Attach the JWT as a Bearer token so the request is authenticated.
def _inject_jwt(request, **kwargs):
    request.headers["Authorization"] = f"Bearer {jwt}"

agentcore_data_client.meta.events.register(
    "before-send.bedrock-agentcore.InvokeAgentRuntime", _inject_jwt
)
try:
    response = agentcore_data_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        payload=json.dumps({"prompt": "How do I get on the corporate Wi-Fi?"}),
    )
finally:
    agentcore_data_client.meta.events.unregister(
        "before-send.bedrock-agentcore.InvokeAgentRuntime", _inject_jwt
    )

# Read the response
print("\n✅ AgentCore Identity validated the JWT and forwarded the request to the agent.")
print("\n🤖 Agent response:")
print(read_agent_response(response))
print("\n" + "=" * 60)

### Task 4.6: Test With Another Prompt

In this task, you send a different question to confirm the agent works correctly through the Runtime.

In [ ]:
print("💬 Testing with another prompt...\n")
print("-" * 60)

agentcore_data_client.meta.events.register(
    "before-send.bedrock-agentcore.InvokeAgentRuntime", _inject_jwt
)
try:
    response = agentcore_data_client.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        payload=json.dumps({"prompt": "My laptop won't power on at all — I've tried holding the power button."}),
    )
finally:
    agentcore_data_client.meta.events.unregister(
        "before-send.bedrock-agentcore.InvokeAgentRuntime", _inject_jwt
    )

print("\n🤖 Agent response:")
print(read_agent_response(response))
print("\n" + "=" * 60)

You confirmed that:
- ❌ Without a JWT → AgentCore Identity rejects the call (agent never sees it)
- ✅ With a valid Cognito JWT → AgentCore Identity validates the JWT and the agent responds

This is the pattern for inbound auth: enforcement happens at the AgentCore Identity layer, not in your application code.

**Task complete.** You verified that AgentCore Identity rejects unauthenticated calls and allows valid ones.

# Task 5: Explore the Cognito User Pool

In this task, you explore the Cognito resources behind your JWT in the AWS Management Console. Run the cell below to print the values that you will reference.

In [ ]:
print("📋 Details for the console walkthrough:\n")
print(f"  Region:                       {REGION}")
print(f"  Cognito user pool ID:         {COGNITO_USER_POOL_ID}")
print(f"  JWT issuer URL:               https://cognito-idp.{REGION}.amazonaws.com/{COGNITO_USER_POOL_ID}")
print(f"  Cognito app client ID:        {COGNITO_CLIENT_ID}")
print(f"  Cognito test user:            {COGNITO_USERNAME}")

Return to the main lab instructions to complete the remainder of Task 5.